# Evaluating the Impact of Image Augmentation Strategies on CNN Architectures in Skin Lesion Classification

**DS 6050, SP 2026 - ML III: Deep Learning Project** *For Professor Heman Shakeri, PhD, University of Virginia*

---

### Team Members
* **Robert Ashby** | *University of Virginia, School of Data Science* | Fernandina Beach, Florida | gsr3qz@virginia.edu
* **Xavier Colbert** | *University of Virginia, School of Data Science* | Alexandria, Virginia | kxp3jj@virginia.edu
* **Jacob Kuchta** | *University of Virginia, School of Data Science* | Arlington, Virginia | mjk3ku@virginia.edu
* **Alysa Pugmire** | *University of Virginia, School of Data Science* | Richmond, Virginia | amp3xs@virginia.edu

# Notebook 06, Cumulative Extendation Validation Study

## Notebook purpose

This notebooks runs the **external evaluation study** for the skin lesion classificatoin project.

Where `04_ablation_study.ipynb` evaluates the effecting of adding augmentation phases sequentially for each backbone by training the backbones on a subset of the ISIC 2019 data and validating the performance with a hold-out test set, this notebook evaluated the effect of adding audmentation phases sequentially using an **external validation dataset** the MILK10k data.

The cumulative abalation ladder is the same as the 04_ablation_study.

The goal is to determine whether the effects of augmentation gains persiste over cross-dataset distribution shifts.

## Execution struccture

This notebook supports two execution modes:

### 1. Smoke-test mode

Used to verify that the full abalation pipeline works end to end on a stratified subset of the MILK10 dataset.

### 2. Full mode

Used to run the full MILK10 validation study.

Because HPC access windows can be limited, this full experiment is designed to run **one backbone at a time**, rather thn forcing all three CNN backbones through the full ablation ladder in a single session.

That means a typical full run now looks like:

- Session 1: ResNet-50 across all cumulative stages
- Session 2: EfficientNet-B0 across all cumulative stages
- Session 3: DenseNet-121 across all cumulative stages
- Final aggregation session: combine per-backbone results and generate summary plots

This keeps the scientific design unchanged while making the notebook practical for short HPC sessions.

In [ ]:
SMOKE_MODE = True

ABLATION_STAGES = [
    "baseline",
    "phase1_plus_2",
    "phase1_plus_2_plus_3",
    "phase1_plus_2_plus_3_plus_4",
]

BACKBONES = [
    "resnet50",
    "efficientnet_b0",
    "densenet121",
]

SMOKE_TRAIN_FRACTION = 0.125
SMOKE_VAL_FRACTION = 0.25
SMOKE_MIN_TRAIN_PER_CLASS = 40
SMOKE_MIN_VAL_PER_CLASS = 15
SMOKE_MAX_TRAIN_PER_CLASS = 250
SMOKE_MAX_VAL_PER_CLASS = 75

IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
FREEZE_FEATURES = False
SAVE_CHECKPOINTS = True

SMOKE_EPOCHS = 1
FULL_EPOCHS = 5
EPOCHS = SMOKE_EPOCHS if SMOKE_MODE else FULL_EPOCHS

EXPERIMENT_SEED = 42

## Imports and project-root setup

This section imports the core Python, PyTorch, and evaluation libraries used throughout the notebook, then resolves the project root so the notebook can consistently find the shared `scripts/` directory and project outputs.

The goal here is to make the notebook portable across local and HPC environments without changing any downstream experiment logic.

In [ ]:
from pathlib import Path
import sys
import json
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms

from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

In [ ]:
# Resolve the project root, not just the current notebook directory
cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name.lower() == "code" else cwd
SCRIPTS = ROOT / "scripts"

if not SCRIPTS.exists():
    raise FileNotFoundError(f"Expected scripts directory at: {SCRIPTS}")

if str(SCRIPTS) not in sys.path:
    sys.path.append(str(SCRIPTS))

from resnet50_baseline import build_resnet50
from efficientnet_b0_baseline import build_efficientnet_b0
from densenet121_baseline import build_densenet121

from phase2_geometric import build_phase2_transform
from phase3_color import build_phase3_transform
from phase4_scale_crop import build_phase4_transform, cutmix_data, cutmix_loss

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("CWD:", cwd)
print("ROOT:", ROOT)
print("SCRIPTS:", SCRIPTS)
print("DEVICE:", DEVICE)

## Save experiment configuration and set seeds

This section creates the output directories for the notebook, saves the active experiment configuration, and fixes the random seed for reproducibility.

Because the setup section is run once at the top of the notebook, the saved configuration should describe the overall notebook environment, not a single backbone-specific HPC session. Backbone-specific execution choices will be handled later in the run cells.

In [ ]:
OUT_ROOT = ROOT / "outputs"

CONFIG_DIR = OUT_ROOT / "configs" / "06_external_evaluation"
FIG_DIR = OUT_ROOT / "figures" / "06_external_evaluation"
METRIC_DIR = OUT_ROOT / "metrics"
PRED_DIR = OUT_ROOT / "preds"
CKPT_DIR = OUT_ROOT / "checkpoints"
REPORT_DIR = OUT_ROOT / "reports" / "06_external_evaluation"

for d in [CONFIG_DIR, FIG_DIR, METRIC_DIR, PRED_DIR, CKPT_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EXPERIMENT_CONFIG = {
    "notebook": "06_external_evaluation",
    "seed": EXPERIMENT_SEED,
    "smoke_mode": SMOKE_MODE,
    "ablation_stages": ABLATION_STAGES,
    "backbones": BACKBONES,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "freeze_features": FREEZE_FEATURES,
    "save_checkpoints": SAVE_CHECKPOINTS,
    "num_workers": NUM_WORKERS,
}

config_path = CONFIG_DIR / ("smoke_config.json" if SMOKE_MODE else "full_config.json")

with open(config_path, "w") as f:
    json.dump(EXPERIMENT_CONFIG, f, indent=2)

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(EXPERIMENT_SEED)

loader_generator = torch.Generator()
loader_generator.manual_seed(EXPERIMENT_SEED)

print("Saved config to:", config_path)
print("Global seed set.")
print("DataLoader generator seeded.")

## Locate and load processed MILK10  splits

This section loads the processed train and validation splits prepared earlier in the project pipeline.

The notebook expects the processed CSV files and class-weight JSON file to already exist under `DATA/processed/`. A small path-normalization helper is used so the image paths behave consistently across Windows, Linux, and Rivanna environments.